# GSB 5544 — Topic 10.1: Model Selection and Hyperparameter Tuning  
*Which features? Which k? Which model? One rule — compare candidates on cross-validated test RMSE, with the same folds for everyone — and one tool that applies it for you: `GridSearchCV`. Fill each `____` blank as you work; the ✅ checks and the review questions ask for a sentence or a number.*

## The plan

Last week: training error flatters, so we estimate the **test error** with K-fold cross-validation. This week
we *use* that estimate to make every choice a model needs, and we let scikit-learn run the loops.

| | | tools |
|---|---|---|
| **1. The rule, and two helpers** | one scorer, one preprocessing builder, used for every candidate | `KFold`, `cross_val_score`, `ColumnTransformer` |
| **2. Model selection** | which *features*? a loop over candidate feature sets | a dictionary of lists |
| **3. Tuning k by hand** | which *k*? last week's loop | `KNeighborsRegressor(n_neighbors=k)` |
| **4. `GridSearchCV`** | the loop, packaged: grid → fit → best | `param_grid`, `best_params_`, `best_score_`, `cv_results_` |
| **5. Several parameters at once** | k × metric × weights; how many fits is that? | a grid with several keys |
| **6. Comparing model families** | KNN vs tree vs line, same folds | one leaderboard |
| **7. Use the winner** | `best_estimator_` is already refitted | `.predict()` |
| **8. Review questions** | | |

**Data:** the Ames houses again (2,930 sales), predicting `SalePrice` in dollars. PA 10.1 uses the same data,
the same tools and the same conventions, with different feature sets — so everything below is a template you
can reuse directly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV

---
## 1. The rule, and two helpers

**The rule.** Every decision — which features, which k, which distance, which model — is made by comparing the
candidates' **cross-validated test RMSE**, computed on the **same folds**. Training RMSE is never consulted: it
always votes for the most flexible candidate (Topic 9.2).

Two helpers keep every comparison identical:

- `folds`: one `KFold` object, created once and passed everywhere. Shuffled, with a `random_state`, so the
  folds are random but repeatable.
- `preprocessing(numeric, categorical)`: builds the `ColumnTransformer` a pipeline needs — a median imputer
  (two Ames columns have one missing value), a scaler for the numeric columns (KNN measures distance), and a
  one-hot encoder with `handle_unknown="ignore"` for the categorical ones (so a rare neighborhood in a
  validation fold does not crash the fold; Topic 9.2 / PA 9.2).

In [ ]:
df_ames = pd.read_csv("https://raw.githubusercontent.com/kevindavisross/data301/main/data/AmesHousing.txt", sep="\t")
y = df_ames["SalePrice"]

folds = KFold(n_splits=____, shuffle=True, random_state=5544)     # the same 5 folds for every candidate

def preprocessing(numeric, categorical=(), scale=True):
    """Impute + (optionally) scale the numeric columns; one-hot encode the categorical ones."""
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        numeric_steps.append(("scale", ____()))
    transformers = [("num", Pipeline(numeric_steps), list(numeric))]
    if categorical:
        transformers.append(("cat", OneHotEncoder(handle_unknown=____), list(categorical)))
    return ColumnTransformer(transformers)

def cv_rmse(model, columns):
    """Cross-validated test RMSE (dollars) of a model on the given columns, on the shared folds."""
    scores = cross_val_score(model, df_ames[columns], y, cv=folds, scoring=____)
    return -scores.mean()

print(df_ames.shape)

✅ **Checkpoint 1.** (a) Why create `folds` once instead of writing `cv=5` in every call? (b) Why does the
scorer's name start with `neg_`, and what does `cv_rmse` do about it?

**Your answer:** *(write it here — replace this line)*

---
## 2. Model selection: a loop over feature sets

"Model selection" here means choosing **which columns** the model gets. Put the candidate feature sets in a
dictionary (name → list of columns), loop, score each with the *same* model and the *same* folds, and read off
the smallest RMSE. Note the model is a 10-nearest-neighbor pipeline every time; only the columns change.

In [ ]:
feature_sets = {
    "size only":                             (["Gr Liv Area"], []),
    "size + quality":                        (["Gr Liv Area", "Overall Qual"], []),
    "size + quality + year":                 (["Gr Liv Area", "Overall Qual", "Year Built"], []),
    "size + quality + year + Neighborhood":  (["Gr Liv Area", "Overall Qual", "Year Built"], ["Neighborhood"]),
}

results = {}
for name, (numeric, categorical) in feature_sets.items():
    pipe = Pipeline([("prep", preprocessing(numeric, categorical)),
                     ("knn", KNeighborsRegressor(n_neighbors=____))])
    results[name] = cv_rmse(pipe, numeric + categorical)

pd.Series(results, name="CV test RMSE (10-NN)").round(0).to_frame()

Each added feature helps, and quality helps the most: knowing `Overall Qual` cuts the error from about 57,900
to 36,500. The last set, **size + quality + year + Neighborhood**, wins at about **33,000** and is the feature
set used for the rest of this notebook.

✅ **Checkpoint 2.** (a) What is the only thing that changes from one loop iteration to the next? (b) Suppose
"size only" had the lowest *training* RMSE. Would that change the decision?

**Your answer:** *(write it here — replace this line)*

---
## 3. Tuning k by hand: last week's loop

A **hyperparameter** is a setting you choose before fitting (k, `max_depth`, the distance metric). Choosing
it is the same game as choosing features: try candidates, score each on the shared folds, take the smallest.
Topic 9.2 did this with a loop. Here it is once more, so that `GridSearchCV` in the next section is recognisably
the same thing.

In [ ]:
numeric, categorical = feature_sets["size + quality + year + Neighborhood"]
columns = numeric + categorical

by_hand = {}
for k in [1, 3, 5, 10, 20, 50]:
    pipe = Pipeline([("prep", preprocessing(numeric, categorical)),
                     ("knn", KNeighborsRegressor(n_neighbors=____))])
    by_hand[k] = cv_rmse(pipe, columns)

pd.Series(by_hand, name="CV test RMSE").round(0).to_frame()

The usual U: k = 1 is noisy (≈ 36,700), the error bottoms out around k = 10 (≈ 33,000), and k = 50 averages
over too many unlike houses (≈ 34,300). The loop works, but it means writing the loop every time, and with two
or three hyperparameters it becomes nested loops. Hence the tool.

---
## 4. `GridSearchCV`: the loop, packaged

`GridSearchCV(model, param_grid, cv, scoring)` does exactly what the loop above did, for every combination of
settings in `param_grid`, and keeps the results:

| argument / attribute | what it is |
|---|---|
| `model` | the pipeline (or model) to tune, **unfitted** |
| `param_grid` | a dictionary: parameter name → list of values to try |
| `cv`, `scoring` | the folds and the scorer, same as `cross_val_score` |
| `.fit(X, y)` | runs the whole search: for every combination, fit on K−1 folds and score on the held-out fold, K times; then **refits the best combination on all of `X`** |
| `.best_params_` | the winning combination |
| `.best_score_` | its mean cross-validated score — **negative**, because of `neg_` |
| `.cv_results_` | a dictionary with every combination's mean and standard deviation of the fold scores, ranks, timings; `pd.DataFrame(...)` makes it readable |
| `.best_estimator_` | the winning pipeline, already refitted on all the rows; use it to predict |

**The naming convention.** The parameter lives inside a pipeline step, so its name is
`<step name>__<parameter>` with a **double underscore**: our KNN step is called `"knn"`, so k is
`"knn__n_neighbors"`. (With `make_pipeline`, steps are named after their class in lower case:
`"kneighborsregressor__n_neighbors"`.) A typo here is the most common `GridSearchCV` error; the message lists
the valid names.

In [ ]:
pipe = Pipeline([("prep", preprocessing(numeric, categorical)),
                 ("knn", KNeighborsRegressor())])            # no k yet: the grid supplies it

param_grid = {____: range(1, 26)}

grid = GridSearchCV(pipe, param_grid, cv=____, scoring="neg_root_mean_squared_error")
grid.____(df_ames[columns], y)

print("best parameters:", grid.best_params_)
print("best CV test RMSE:", round(-grid.____))

`cv_results_` holds the whole search. The columns to know: `param_knn__n_neighbors` (the setting),
`mean_test_score` (mean over folds, negative RMSE), `std_test_score` (spread over folds), `rank_test_score`.
A plot of RMSE against k is the picture that justifies the choice.

In [ ]:
cv_table = pd.DataFrame(grid.cv_results_)
cv_table["CV test RMSE"] = -cv_table[____]
cv_table[["param_knn__n_neighbors", "CV test RMSE", "std_test_score", "rank_test_score"]].round(0).head(12)

In [ ]:
ax = cv_table.plot(x="param_knn__n_neighbors", y="CV test RMSE", marker="o", legend=False)
ax.axvline(grid.best_params_["knn__n_neighbors"], color="firebrick", linestyle="--")
ax.set_xlabel("k (n_neighbors)"); ax.set_ylabel("cross-validated test RMSE ($)")
ax.set_title("GridSearchCV over k");

✅ **Checkpoint 3.** (a) The grid has 25 values of k and `folds` has 5 folds. How many models did `grid.fit`
fit, and why is it one more than 25 × 5? (b) `grid.best_score_` is about −32,900. What is the test RMSE
estimate, and in what units? (c) k = 7, 8, 9 and 10 are all within about 100 dollars of each other while the
fold-to-fold standard deviation is about 2,700. How confident should you be that k = 8 is really best?

**Your answer:** *(write it here — replace this line)*

---
## 5. Several parameters at once

Add keys to `param_grid` and `GridSearchCV` tries **every combination** (the "grid"). `KNeighborsRegressor` has
more than k to tune:

- `metric`: how distance is measured — `"euclidean"` (straight line, the default) or `"manhattan"` (sum of
  absolute differences along each feature).
- `weights`: `"uniform"` averages the k neighbors equally; `"distance"` gives closer neighbors more weight.

The cost: the number of fits is (grid size) × (folds). 25 values of k × 2 metrics = 50 combinations × 5 folds
= 250 fits; add 2 weightings and it is 500. Still seconds here, but grids grow fast.

In [ ]:
param_grid_2 = {
    "knn__n_neighbors": range(1, 26),
    ____: ["euclidean", "manhattan"],
}
grid2 = GridSearchCV(pipe, param_grid_2, cv=folds, scoring="neg_root_mean_squared_error")
grid2.fit(df_ames[columns], y)

print("combinations tried:", len(grid2.cv_results_["params"]), "   fits:", len(grid2.cv_results_["params"]) * folds.get_n_splits())
print("best:", grid2.best_params_, "  CV test RMSE:", round(-grid2.best_score_))

With two parameters the natural picture is a table or two curves: `pivot` the results so that rows are k and
columns are the metric.

In [ ]:
table2 = pd.DataFrame(grid2.cv_results_)
table2["CV test RMSE"] = -table2["mean_test_score"]
wide = table2.pivot(index="param_knn__n_neighbors", columns="param_knn__metric", values="CV test RMSE")
ax = wide.plot(marker="o"); ax.set_xlabel("k"); ax.set_ylabel("cross-validated test RMSE ($)")
wide.round(0).head(12)

In [ ]:
param_grid_3 = {
    "knn__n_neighbors": range(1, 26),
    "knn__metric": ["euclidean", "manhattan"],
    ____: ["uniform", "distance"],
}
grid3 = GridSearchCV(pipe, param_grid_3, cv=folds, scoring="neg_root_mean_squared_error")
grid3.fit(df_ames[columns], y)
print("fits:", len(grid3.cv_results_["params"]) * folds.get_n_splits())
print("best:", grid3.best_params_, "  CV test RMSE:", round(-grid3.best_score_))

✅ **Checkpoint 4.** (a) Manhattan distance with k = 6 beat Euclidean with k = 8 by about 70 dollars. Is the
metric worth changing? (b) `weights="distance"` brought the RMSE down to about 31,000. In one sentence, why
might weighting by distance help KNN on house prices? (c) You want to also try `p` in {1, 2, 3} with
`metric="minkowski"`. How many fits would a grid over k (25), weights (2) and p (3) need with 5 folds?

**Your answer:** *(write it here — replace this line)*

---
## 6. Comparing model families on the same folds

Hyperparameter tuning picks the best *version* of a model. Model selection in the wide sense also compares
*kinds* of model. Same rule: every family gets its best settings by grid search (or needs none), and the
families are compared by cross-validated test RMSE on the same folds. The tree's grid is over `max_depth`; the
tree needs no scaling, so `preprocessing(..., scale=False)` keeps the thresholds in real units.

In [ ]:
tree_pipe = Pipeline([("prep", preprocessing(numeric, categorical, scale=False)),
                      ("tree", DecisionTreeRegressor(random_state=5544))])
grid_tree = GridSearchCV(tree_pipe, {____: [2, 4, 6, 8, 10, 12, None]},
                         cv=folds, scoring="neg_root_mean_squared_error")
grid_tree.fit(df_ames[columns], y)

linear_pipe = Pipeline([("prep", preprocessing(numeric, categorical)), ("lm", LinearRegression())])

leaderboard = pd.Series({
    f"KNN {grid3.best_params_}":            -grid3.best_score_,
    f"tree {grid_tree.best_params_}":       -grid_tree.best_score_,
    "linear regression (nothing to tune)":  cv_rmse(linear_pipe, columns),
}, name="CV test RMSE").sort_values().round(0).to_frame()
leaderboard

✅ **Checkpoint 5.** (a) Why is it fair to compare `grid3.best_score_` with `cv_rmse(linear_pipe, columns)`?
(b) The tuned tree and the line are within a few hundred dollars of each other at about 35,500. Which would
you choose, and on what grounds other than RMSE?

**Your answer:** *(write it here — replace this line)*

---
## 7. Use the winner

`best_estimator_` is the winning pipeline **already refitted on all 2,930 houses** — exactly the "refit the
chosen model on all the data" step from Topic 9.2, done for you. Predict with it directly.

In [ ]:
new_house = pd.DataFrame({"Gr Liv Area": [1500], "Overall Qual": [7], "Year Built": [2005], "Neighborhood": ["CollgCr"]})
best_knn = grid3.____
print("predicted SalePrice:", best_knn.predict(new_house).round(0))
best_knn

✅ **Checkpoint 6.** `grid3.predict(new_house)` gives the same number as `best_knn.predict(new_house)`. Why,
and which rows was that model fitted on?

**Your answer:** *(write it here — replace this line)*

---
## 8. Review questions

Short answers; a sentence or a number each.

**Q1.** State the one rule behind every choice in this notebook, and name the quantity that is never used.

**Your answer:** *(write it here — replace this line)*

**Q2.** A pipeline has steps `("prep", ...)` and `("model", KNeighborsRegressor())`. Write the `param_grid`
that tries k = 5, 10, 15 and both weightings. How many fits with `KFold(10)`?

**Your answer:** *(write it here — replace this line)*

**Q3.** `grid.best_score_` prints `-41237.5`. What does it mean?

**Your answer:** *(write it here — replace this line)*

**Q4.** What is the difference between `grid.cv_results_` and `grid.best_estimator_`?

**Your answer:** *(write it here — replace this line)*

**Q5.** Two values of k differ by 50 dollars in mean CV RMSE; the standard deviation across folds is 3,000.
What do you report?

**Your answer:** *(write it here — replace this line)*

---
## The lines to keep

| | |
|---|---|
| **Shared folds** | `folds = KFold(n_splits=5, shuffle=True, random_state=5544)` — pass the same object to every `cross_val_score` and `GridSearchCV` |
| **Feature selection** | a dict `name → columns`; loop; `cv_rmse(pipeline, columns)`; smallest wins |
| **Grid search** | `GridSearchCV(pipe, {"knn__n_neighbors": range(1, 26)}, cv=folds, scoring="neg_root_mean_squared_error").fit(X, y)` |
| **Names** | `"<step>__<parameter>"` — `knn__n_neighbors`, `knn__metric`, `knn__weights`, `tree__max_depth` |
| **Results** | `best_params_`; `-best_score_` (RMSE); `pd.DataFrame(cv_results_)` → `mean_test_score`, `std_test_score`, `rank_test_score`; plot RMSE vs the parameter |
| **Cost** | fits = (number of combinations) × (folds) + 1 |
| **Use it** | `best_estimator_` is refitted on all rows: `grid.predict(new)` |

Course site: [https://gato365.github.io/gsb5544_instructor_learn_prep/](https://gato365.github.io/gsb5544_instructor_learn_prep/).